In [28]:
from pathlib import Path
import configparser
import os
import subprocess
import sys

import imageio.v2 as imageio
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

here = Path.cwd().resolve()
REPO_ROOT = next((p for p in [here, *here.parents] if (p / 'python_unix').is_dir()), None)
if REPO_ROOT is None:
    raise FileNotFoundError('Could not find the repository root containing python_unix/')

PYTHON_UNIX = REPO_ROOT / 'python'
RUN_DATA = REPO_ROOT / 'sample_data'
sys.path.insert(0, str(PYTHON_UNIX))

from ebiv_params import EBIV_Config
from ebiv_utils import EBISaveEvents

In [ ]:
# Replace this with your NPZ path.
EVENT_NPZ = Path('E:\\cvis_att1\\events\\no_drone_w_lightsheet.npz')
EVENT_TIME_UNIT = 's'  # events_from_exr.py stores t in seconds.
EVENT_STEM = EVENT_NPZ.stem

if not EVENT_NPZ.is_file():
    raise FileNotFoundError(f'Update EVENT_NPZ first: {EVENT_NPZ}')

events_npz = np.load(EVENT_NPZ)
required = {'x', 'y', 't', 'p', 'height', 'width'}
missing = required.difference(events_npz.files)
if missing:
    raise KeyError(f'Missing NPZ arrays: {sorted(missing)}; found {events_npz.files}')

event_x = np.asarray(events_npz['x']).reshape(-1)
event_y = np.asarray(events_npz['y']).reshape(-1)
event_t_s = np.asarray(events_npz['t'], dtype=np.float64).reshape(-1)
if EVENT_TIME_UNIT == 's':
    event_t = np.rint(event_t_s * 1_000_000).astype(np.int64)
elif EVENT_TIME_UNIT == 'us':
    event_t = np.rint(event_t_s).astype(np.int64)
elif EVENT_TIME_UNIT == 'ns':
    event_t = np.rint(event_t_s / 1_000).astype(np.int64)
else:
    raise ValueError("EVENT_TIME_UNIT must be 's', 'us', or 'ns'")
event_t = event_t - event_t.min()  # EBIV EVT timestamps are integer microseconds.
event_p = np.asarray(events_npz['p']).reshape(-1)
sensor_height = int(np.asarray(events_npz['height']).reshape(()))
sensor_width = int(np.asarray(events_npz['width']).reshape(()))
events_npz.close()

if not (event_x.size == event_y.size == event_t.size == event_p.size):
    raise ValueError('x, y, t, and p must contain the same number of events')
if event_x.size == 0:
    raise ValueError('The event arrays are empty')
if sensor_height <= 0 or sensor_width <= 0:
    raise ValueError('height and width must be positive')

print(f'Loaded {event_x.size:,} events from {EVENT_NPZ}')
print(f'Event basename: {EVENT_STEM}')
print(f'Sensor size: {sensor_width} x {sensor_height}')
print(f'Time range: {event_t.min() / 1_000_000:.6f} to {event_t.max() / 1_000_000:.6f} s')
print(f'Duration: {(event_t.max() - event_t.min()) / 1_000_000:.6f} s')

Loaded 1,112,489 events from E:\cvis_att1\events\no_drone_normal_lighting.npz
Event basename: no_drone_normal_lighting
Sensor size: 640 x 360
Time range: 0.000000 to 9.970566 s
Duration: 9.970566 s


In [30]:
EVENT_FRAME_DURATION_US = 20_000  # 20 ms per frame = 50 fps
EVENT_VIDEO_FPS = 1_000_000 / EVENT_FRAME_DURATION_US
EVENT_VIDEO_FILE = RUN_DATA / 'plots' / f'{EVENT_STEM}_events.mp4'
EVENT_VIDEO_FILE.parent.mkdir(parents=True, exist_ok=True)

input_duration_us = int(event_t.max() - event_t.min())
frame_starts = np.arange(0, input_duration_us, EVENT_FRAME_DURATION_US)

with imageio.get_writer(str(EVENT_VIDEO_FILE), fps=EVENT_VIDEO_FPS, codec='libx264') as writer:
    for frame_start in frame_starts:
        frame_stop = frame_start + EVENT_FRAME_DURATION_US
        frame_mask = (event_t >= frame_start) & (event_t < frame_stop)
        positive = frame_mask & (event_p > 0)
        negative = frame_mask & (event_p <= 0)

        fig, ax = plt.subplots(figsize=(6.4, 4.8), dpi=100)
        ax.scatter(event_x[positive], event_y[positive], s=1, c='red', label='positive')
        ax.scatter(event_x[negative], event_y[negative], s=1, c='blue', label='negative')
        ax.set_xlim(0, sensor_width)
        ax.set_ylim(sensor_height, 0)
        ax.set_aspect('equal')
        ax.set_title(f'Events: {frame_start / 1000:.1f} ms')
        ax.set_xlabel('X [pixel]')
        ax.set_ylabel('Y [pixel]')
        fig.tight_layout()
        fig.canvas.draw()
        frame = np.asarray(fig.canvas.buffer_rgba())[..., :3]
        writer.append_data(frame)
        plt.close(fig)

print(f'Created event video: {EVENT_VIDEO_FILE}')
print(f'Frames: {len(frame_starts)}')
print(f'Duration: {len(frame_starts) / EVENT_VIDEO_FPS:.3f} s')

Created event video: E:\cvis_att1\ebiv\sample_data\plots\no_drone_normal_lighting_events.mp4
Frames: 499
Duration: 9.980 s


# EBIV processing from an event NPZ

This notebook adapts an NPZ containing `x`, `y`, `t`, `p`, `height`, and `width` into EBIV's EVT format, then calls the existing scripts in `python_unix`. It does not reimplement velocity estimation or plotting.

Set `EVENT_NPZ` in the next cell. The staged run directory is separate from the repository's existing `sample_data` files.

## Convert the NPZ to EBIV EVT

The existing processing script reads `.evt` or `.raw`. This cell uses the repository's existing `EBISaveEvents` helper rather than implementing the binary format here.

In [31]:
RUN_DATA.mkdir(parents=True, exist_ok=True)
evt_file = RUN_DATA / f'{EVENT_STEM}.evt'
event_dict = {
    't': event_t.astype(np.uint64),
    'x': event_x.astype(np.uint32),
    'y': event_y.astype(np.uint32),
    'p': (event_p > 0).astype(np.uint32),
    'time_stamp': 0,
    'image_size': [sensor_height, sensor_width],
}
evt_file.unlink(missing_ok=True)
EBISaveEvents(str(evt_file), event_dict)
print(f'Overwrote {evt_file}')

Wrote 1112489 events of duration 9970.57ms into EVT file with (64+8899912) bytes
Overwrote E:\cvis_att1\ebiv\sample_data\no_drone_normal_lighting.evt


## Configure the existing scripts

The repository config and generated EVT stay in `sample_data`. The original modules remain in `python_unix`; no staging directory or copied scripts is used.

In [32]:
# Native dense EBIV settings: every 10 px, using 40x40-pixel event windows.
# This computes denser measurements directly; no spatial interpolation is applied.
PREVIEW_END_US = int(event_t.max())
PREVIEW_STEP_US = 20_000
PREVIEW_SAMPLE_WIDTH = 40
PREVIEW_SAMPLE_HEIGHT = 40
PREVIEW_SAMPLE_STEP = 10
PREVIEW_STEM = f'{EVENT_STEM}_native_dense10'

config_source = RUN_DATA / 'ebiv_params.cfg'
config = configparser.ConfigParser()
config.read(config_source)
config['DEFAULT']['FileStub'] = PREVIEW_STEM
config['DEFAULT']['ImageWidth'] = str(sensor_width)
config['DEFAULT']['ImageHeight'] = str(sensor_height)
config['DEFAULT']['DoSingleTimeStep'] = 'False'
config['DEFAULT']['SampleTimeStart'] = '0'
config['DEFAULT']['SampleTimeEnd'] = str(PREVIEW_END_US)
config['DEFAULT']['SampleTimeStep'] = str(PREVIEW_STEP_US)
config['DEFAULT']['SampleDuration'] = str(PREVIEW_STEP_US)
config['DEFAULT']['SampleWidth'] = str(PREVIEW_SAMPLE_WIDTH)
config['DEFAULT']['SampleHeight'] = str(PREVIEW_SAMPLE_HEIGHT)
config['DEFAULT']['SampleStepX'] = str(PREVIEW_SAMPLE_STEP)
config['DEFAULT']['SampleStepY'] = str(PREVIEW_SAMPLE_STEP)
with config_source.open('w') as config_file:
    config.write(config_file)

print(f'Configured {config_source} for {EVENT_STEM}.evt')
print(f'Input and output directory: {RUN_DATA}')
print(f'Native output stem: {PREVIEW_STEM}')
print(f'Windows: {config["DEFAULT"]["SampleTimeEnd"]} us, '
      f'{PREVIEW_STEP_US} us cadence; spatial grid every {PREVIEW_SAMPLE_STEP}px '
      f'with {PREVIEW_SAMPLE_WIDTH}x{PREVIEW_SAMPLE_HEIGHT}px windows')

Configured E:\cvis_att1\ebiv\sample_data\ebiv_params.cfg for no_drone_normal_lighting.evt
Input and output directory: E:\cvis_att1\ebiv\sample_data
Native output stem: no_drone_normal_lighting_native_dense10
Windows: 9970566 us, 20000 us cadence; spatial grid every 10px with 40x40px windows


## Run velocity estimation and plotting

These are direct subprocess calls to the existing scripts. Processing settings are controlled by `RUN_DATA/ebiv_params.cfg`; edit the repository config or add parameter edits above if needed.

In [38]:
PLOT_ARROW_SCALE = 100  # smaller values make arrows longer


def run_script(script_name, arguments=()):
    print(f'Notebook Python: {sys.executable}', flush=True)
    try:
        import cupy as cp
        device_name = cp.cuda.runtime.getDeviceProperties(0)['name'].decode()
        print(f'CuPy: {cp.__version__}; CUDA devices: {cp.cuda.runtime.getDeviceCount()}; GPU: {device_name}', flush=True)
    except Exception as exc:
        print(f'CuPy/CUDA unavailable in notebook kernel: {exc}', flush=True)
    command = [sys.executable, '-u', str(PYTHON_UNIX / script_name), *arguments]
    environment = os.environ.copy()
    environment['PYTHONUNBUFFERED'] = '1'
    log_file = RUN_DATA / f'{Path(script_name).stem}_live.log'
    print(f'Running {script_name} with unbuffered output...', flush=True)
    print(f'Live log: {log_file}', flush=True)
    with log_file.open('w', encoding='utf-8') as log:
        process = subprocess.Popen(
            command,
            cwd=PYTHON_UNIX,
            env=environment,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end='', flush=True)
            log.write(line)
            log.flush()
        return_code = process.wait()
    if return_code:
        raise subprocess.CalledProcessError(return_code, command)
    return return_code

In [34]:
print(config_source)
print(evt_file)
print(RUN_DATA)
run_script('ebiv_process.py', [
    '--config', str(config_source),
    '--events', str(evt_file),
    '--output-dir', str(RUN_DATA),
])

E:\cvis_att1\ebiv\sample_data\ebiv_params.cfg
E:\cvis_att1\ebiv\sample_data\no_drone_normal_lighting.evt
E:\cvis_att1\ebiv\sample_data
Notebook Python: c:\Users\ryley\.conda\envs\smoke-vdb12\python.exe
CuPy: 14.2.0; CUDA devices: 1; GPU: NVIDIA GeForce RTX 3080
Running ebiv_process.py with unbuffered output...
Live log: E:\cvis_att1\ebiv\sample_data\ebiv_process_live.log
CUDA acceleration: enabled (1 CUDA device(s) detected)
Loading events from: E:\cvis_att1\ebiv\sample_data\no_drone_normal_lighting.evt
Window length: 20.000 ms
Input file duration: 9970.566 ms
Processing 499 windows (2013 positions/window, 1004487 total positions)

  Current window:  10.0% (201/2013 samples)
  Current window:  20.0% (402/2013 samples)
  Current window:  30.0% (603/2013 samples)
  Current window:  39.9% (804/2013 samples)
  Current window:  49.9% (1005/2013 samples)
  Current window:  59.9% (1206/2013 samples)
  Current window:  69.9% (1407/2013 samples)
  Current window:  79.9% (1608/2013 samples)
  Cu

0

In [44]:
import re

velocity_stem = config['DEFAULT']['FileStub']
velocity_dir = RUN_DATA / 'velocity_fields' / velocity_stem
window_files = sorted(velocity_dir.glob(f'{velocity_stem}_t*us.npz'))
if not window_files:
    raise FileNotFoundError(f'No per-window velocity files found in {velocity_dir}')

sample_duration_us = config.getint('DEFAULT', 'SampleDuration', fallback=20000)
sample_step_us = config.getint('DEFAULT', 'SampleTimeStep', fallback=sample_duration_us)
fps = 1_000_000.0 / sample_step_us if sample_step_us > 0 else 50.0

velocity_arrays = []
t_starts_list = []
t_ends_list = []
bbox = np.asarray([], dtype=np.int32)
image_size = np.asarray([sensor_height, sensor_width], dtype=np.int32)
for window_file in window_files:
    match = re.search(r'_t(\d+)us', window_file.name)
    if not match:
        raise ValueError(f'Cannot parse timestamp from {window_file.name}')
    t_0_us = int(match.group(1))
    t_starts_list.append(t_0_us / 1_000_000.0)
    t_ends_list.append((t_0_us + sample_duration_us) / 1_000_000.0)

    with np.load(window_file) as window_npz:
        if 'velData' not in window_npz.files:
            raise KeyError(f'Missing velData in {window_file}')
        velocity_arrays.append(np.asarray(window_npz['velData']))
        if 'bbox' in window_npz.files and window_npz['bbox'].size:
            bbox = np.asarray(window_npz['bbox'], dtype=np.int32)
        if 'image_size' in window_npz.files:
            image_size = np.asarray(window_npz['image_size'], dtype=np.int32)

combined_velocity = np.concatenate(velocity_arrays, axis=0)
t_starts = np.asarray(t_starts_list, dtype=np.float64)
t_ends = np.asarray(t_ends_list, dtype=np.float64)
times = (t_starts + t_ends) / 2.0

combined_velocity_file = velocity_dir / f'{velocity_stem}_combined.npz'
np.savez(
    combined_velocity_file,
    velData=combined_velocity,
    bbox=bbox,
    image_size=image_size,
    times=times,
    t_starts=t_starts,
    t_ends=t_ends,
    fps=fps,
)
print(f'Saved combined EBIV results: {combined_velocity_file}')
print(f'Combined velData shape: {combined_velocity.shape}')
print(f'Saved timestamps: {len(times)} frames from {times[0]:.4f}s to {times[-1]:.4f}s (cadence: {fps:.2f} fps)')


Saved combined EBIV results: E:\cvis_att1\ebiv\sample_data\velocity_fields\no_drone_normal_lighting_native_dense10\no_drone_normal_lighting_native_dense10_combined.npz
Combined velData shape: (499, 33, 61, 6)


In [45]:
velocity_stem = config['DEFAULT']['FileStub']
velocity_dir = RUN_DATA / 'velocity_fields' / velocity_stem
window_files = sorted(velocity_dir.glob(f'{velocity_stem}_t*us.npz'))
if not window_files:
    raise FileNotFoundError(f'No per-window velocity files found in {velocity_dir}')

max_velocity_px_ms = 0.0
for velocity_file in window_files:
    with np.load(velocity_file) as velocity_npz:
        if 'velData' not in velocity_npz.files:
            raise KeyError(f'Missing velData in {velocity_file}')
        velocity_data = velocity_npz['velData']
    window_speed = np.hypot(velocity_data[..., 2], velocity_data[..., 3])
    max_velocity_px_ms = max(max_velocity_px_ms, float(np.nanmax(window_speed)))

PLOT_VMAX = max_velocity_px_ms * 1000
if not np.isfinite(PLOT_VMAX) or PLOT_VMAX <= 0:
    raise ValueError('The velocity files contain no positive finite velocity magnitude')
print(f'Maximum velocity across all windows: {PLOT_VMAX:.6g} pixel/s')

Maximum velocity across all windows: 5830.95 pixel/s


In [40]:
run_script('plot_velmap.py', [
    '--vmax', str(1000),
    '--arrow-scale', str(PLOT_ARROW_SCALE),
])

Notebook Python: c:\Users\ryley\.conda\envs\smoke-vdb12\python.exe
CuPy: 14.2.0; CUDA devices: 1; GPU: NVIDIA GeForce RTX 3080
Running plot_velmap.py with unbuffered output...
Live log: E:\cvis_att1\ebiv\sample_data\plot_velmap_live.log
Setting default font to: Times
detected 4 possible outliers
detected 4 possible outliers
detected 4 possible outliers
plot stored in: E:\cvis_att1\ebiv\sample_data\plots\veldata_t0ms_w40_motion_var_res0_25_0000.png
detected 0 possible outliers
detected 0 possible outliers
detected 0 possible outliers
plot stored in: E:\cvis_att1\ebiv\sample_data\plots\veldata_t0ms_w40_motion_var_res0_25_0001.png
detected 0 possible outliers
detected 0 possible outliers
detected 0 possible outliers
plot stored in: E:\cvis_att1\ebiv\sample_data\plots\veldata_t0ms_w40_motion_var_res0_25_0002.png
detected 0 possible outliers
detected 0 possible outliers
detected 0 possible outliers
plot stored in: E:\cvis_att1\ebiv\sample_data\plots\veldata_t0ms_w40_motion_var_res0_25_0003.

0

In [37]:
cfg = EBIV_Config(str(config_source))
plot_stem = cfg.velocityDataFile()
plot_dir = RUN_DATA / 'plots'
plot_files = sorted(plot_dir.glob(f'{plot_stem}_*.png'))
if not plot_files:
    single_plot = plot_dir / f'{plot_stem}.png'
    if single_plot.is_file():
        plot_files = [single_plot]
if not plot_files:
    raise FileNotFoundError(f'No velocity plot PNGs found in {plot_dir}')

processed_duration_us = len(plot_files) * config.getint('DEFAULT', 'SampleTimeStep')
if processed_duration_us <= 0:
    raise ValueError('Processed duration must be positive to create a video')
video_fps = len(plot_files) / (processed_duration_us / 1_000_000)
video_file = plot_dir / f'{EVENT_STEM}_velocity.mp4'

with imageio.get_writer(str(video_file), fps=video_fps, codec='libx264') as writer:
    for plot_file in plot_files:
        writer.append_data(imageio.imread(plot_file))

print(f'Created video: {video_file}')
print(f'Frames: {len(plot_files)}')
print(f'Target duration: {processed_duration_us / 1000:.3f} ms')
print(f'Video frame rate: {video_fps:.6g} fps')

IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (1800, 1000) to (1808, 1008) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


Created video: E:\cvis_att1\ebiv\sample_data\plots\no_drone_normal_lighting_velocity.mp4
Frames: 500
Target duration: 10000.000 ms
Video frame rate: 50 fps


In [ ]:
combined_video_file = plot_dir / f'{EVENT_STEM}_events_and_velocity.mp4'
event_reader = imageio.get_reader(str(EVENT_VIDEO_FILE))
velocity_reader = imageio.get_reader(str(video_file))
event_frame_count = event_reader.count_frames()
velocity_frame_count = velocity_reader.count_frames()
if event_frame_count == 0 or velocity_frame_count == 0:
    event_reader.close()
    velocity_reader.close()
    raise ValueError('Both source videos must contain at least one frame')

combined_fps = velocity_frame_count / (input_duration_us / 1_000_000)
first_event_frame = event_reader.get_data(0)
first_velocity_frame = velocity_reader.get_data(0)
combined_height = min(first_event_frame.shape[0], first_velocity_frame.shape[0])

def resize_to_height(frame, height):
    if frame.shape[0] == height:
        return frame
    width = round(frame.shape[1] * height / frame.shape[0])
    return np.asarray(Image.fromarray(frame).resize((width, height)))

with imageio.get_writer(str(combined_video_file), fps=combined_fps, codec='libx264') as writer:
    for velocity_index in range(velocity_frame_count):
        event_index = round(velocity_index * (event_frame_count - 1) /
                            max(velocity_frame_count - 1, 1))
        event_frame = resize_to_height(event_reader.get_data(event_index), combined_height)
        velocity_frame = resize_to_height(velocity_reader.get_data(velocity_index), combined_height)
        writer.append_data(np.concatenate((event_frame, velocity_frame), axis=1))

event_reader.close()
velocity_reader.close()
print(f'Created side-by-side video: {combined_video_file}')
print(f'Frames: {velocity_frame_count}')
print(f'Target duration: {input_duration_us / 1_000_000:.3f} s')

In [ ]:
frames_dir = combined_video_file.with_name(f'{combined_video_file.stem}_frames')
frames_dir.mkdir(parents=True, exist_ok=True)

video_reader = imageio.get_reader(str(combined_video_file))
try:
    frame_count = 0
    for frame_count, frame in enumerate(video_reader):
        frame_file = frames_dir / f'frame_{frame_count:06d}.png'
        imageio.imwrite(frame_file, frame)
finally:
    video_reader.close()

print(f'Exported {frame_count + 1 if frame_count else 0} frames to {frames_dir}')

In [ ]:
ORIGINAL_VIDEO_FILE = Path('./F01_c01.mp4')  # Update this path to your original video file
original_video_output = combined_video_file.with_name(
    f'{combined_video_file.stem}_with_original.mp4'
)

if not ORIGINAL_VIDEO_FILE.is_file():
    raise FileNotFoundError(f'Update ORIGINAL_VIDEO_FILE first: {ORIGINAL_VIDEO_FILE}')

def to_rgb(frame):
    if frame.ndim == 2:
        return np.repeat(frame[..., None], 3, axis=2)
    return frame[..., :3]


def resize_to_width(frame, width):
    if frame.shape[1] == width:
        return frame
    height = round(frame.shape[0] * width / frame.shape[1])
    return np.asarray(Image.fromarray(frame).resize((width, height)))

combined_reader = imageio.get_reader(str(combined_video_file))
original_reader = imageio.get_reader(str(ORIGINAL_VIDEO_FILE))
try:
    combined_frame_count = combined_reader.count_frames()
    original_frame_count = original_reader.count_frames()
    if combined_frame_count == 0 or original_frame_count == 0:
        raise ValueError('Both videos must contain at least one frame')

    output_fps = combined_frame_count / (input_duration_us / 1_000_000)
    first_combined_frame = to_rgb(combined_reader.get_data(0))
    output_width = first_combined_frame.shape[1]

    with imageio.get_writer(str(original_video_output), fps=output_fps, codec='libx264') as writer:
        for combined_index in range(combined_frame_count):
            original_index = round(
                combined_index * (original_frame_count - 1) /
                max(combined_frame_count - 1, 1)
            )
            combined_frame = to_rgb(combined_reader.get_data(combined_index))
            original_frame = resize_to_width(
                to_rgb(original_reader.get_data(original_index)), output_width
            )
            writer.append_data(np.concatenate((combined_frame, original_frame), axis=0))
finally:
    combined_reader.close()
    original_reader.close()

print(f'Created video with original footage below: {original_video_output}')
print(f'Frames: {combined_frame_count}')